# Connect Four

A small reasoning benchmark for the Azure deployments `gpt-4o`, `gpt-4o-mini`, and `gpt-4.1`.

Chinchilla scaling says a smaller model trained on more tokens can beat a larger model trained on fewer. Parameter count is not the whole choice. Public exams such as GPQA, MMLU-Pro, and HLE score fixed questions. They go stale when those questions leak into training data, and a model can look strong because it memorized the set. The five boards below are the local version of that exam: one correct column each.


## Client

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
azure_model = os.getenv("GPT_MODEL", "gpt-4o")
deployments = []
for name in (azure_model, "gpt-4o-mini", "gpt-4.1"):
    if name not in deployments:
        deployments.append(name)


## Board

Seven columns, six rows. Row 0 is the bottom. `X` moves first.

In [ ]:
COLS = 7
ROWS = 6


def new_board():
    return [["."] * COLS for _ in range(ROWS)]


def drop(board, col, piece):
    if not 0 <= col < COLS or board[ROWS - 1][col] != ".":
        return False
    for row in range(ROWS):
        if board[row][col] == ".":
            board[row][col] = piece
            return True
    return False


def winner(board, piece):
    for row in range(ROWS):
        for col in range(COLS):
            if board[row][col] != piece:
                continue
            for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                cells = []
                for step in range(4):
                    rr, cc = row + dr * step, col + dc * step
                    if not (0 <= rr < ROWS and 0 <= cc < COLS):
                        break
                    cells.append(board[rr][cc])
                if cells == [piece] * 4:
                    return True
    return False


def play(moves):
    board = new_board()
    for i, col in enumerate(moves):
        piece = "X" if i % 2 == 0 else "O"
        drop(board, col, piece)
    return board


def show(board):
    lines = [" ".join(row) for row in reversed(board)]
    lines.append("0 1 2 3 4 5 6")
    return "\n".join(lines)


## Positions

Each tuple is a name, the moves already played, and the correct column for `X`. `choose the win` has a block available and a win available. The win is the right reply.

In [ ]:
positions = [
    ("row", [0, 4, 1, 5, 2], 3),
    ("block", [6, 0, 6, 1, 5, 2], 3),
    ("column", [2, 0, 2, 1, 2, 3], 2),
    ("diagonal", [0, 1, 1, 2, 2, 3, 2, 3, 3, 6], 3),
    ("choose the win", [6, 0, 6, 1, 6, 2], 6),
]

for name, moves, answer in positions:
    print(name, "column", answer)
    print(show(play(moves)))
    print()


## Ask

A winning column beats a block. The reply is one digit.

In [ ]:
system = """
You are X in Connect Four. Columns are numbered 0 to 6.
If one column wins immediately, reply with that column.
If you cannot win this turn, reply with the column that stops O from winning on the next turn.
Reply with one digit and no other text.
"""


def column_from(text):
    for char in text or "":
        if char.isdigit():
            col = int(char)
            return col if col <= 6 else None
    return None


def ask(deployment, board):
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": show(board)},
        ],
        timeout=120.0,
    )
    return column_from(response.choices[0].message.content)


## Leaderboard

In [ ]:
def score(deployment):
    correct = 0
    for name, moves, answer in positions:
        guess = ask(deployment, play(moves))
        ok = guess == answer
        correct += ok
        mark = "ok" if ok else f"wanted {answer}"
        print(f"{deployment}  {name}: {guess}  {mark}")
    print()
    return correct


rows = [(name, score(name)) for name in deployments]
rows.sort(key=lambda row: (-row[1], row[0]))
print("Rank  Model             Score")
for rank, (name, correct) in enumerate(rows, start=1):
    print(f"{rank:<5} {name:<16}  {correct}/{len(positions)}")
